In [1]:
import polars as pl
from FlagEmbedding import BGEM3FlagModel

In [3]:
DATA_DIR  = "../data/ml-25m"   

In [2]:
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

onnx/model.onnx_data:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

In [21]:
movies = pl.read_csv(f"{DATA_DIR}/movies.csv",
                     schema_overrides={
                         "movieId": pl.Int32,
                         "title":   pl.Utf8,
                         "genres":  pl.Utf8
                     })
movies

movieId,title,genres
i32,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…"
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance"""
5,"""Father of the Bride Part II (1…","""Comedy"""
…,…,…
209157,"""We (2018)""","""Drama"""
209159,"""Window of the Soul (2001)""","""Documentary"""
209163,"""Bad Poems (2018)""","""Comedy|Drama"""


In [23]:
soup_strings = (
    movies.select(
        pl.col("title") + " " + pl.col("genres").str.replace_all(r"\|", " ")
    )
    .to_series()
    .to_list()
)

In [24]:
embeddings = model.encode(soup_strings)

Inference Embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 244/244 [00:11<00:00, 22.12it/s]


In [31]:
movies = movies.with_columns(
    embedding = pl.Series(embeddings["dense_vecs"].tolist())
)

In [32]:
movies

movieId,title,genres,embedding
i32,str,str,list[f64]
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","[-0.02388, -0.012184, … 0.007309]"
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy""","[-0.011749, -0.011139, … 0.012016]"
3,"""Grumpier Old Men (1995)""","""Comedy|Romance""","[-0.032593, 0.007195, … 0.001019]"
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance""","[-0.032013, -0.011787, … 0.043274]"
5,"""Father of the Bride Part II (1…","""Comedy""","[-0.003693, 0.034637, … 0.01403]"
…,…,…,…
209157,"""We (2018)""","""Drama""","[0.016785, 0.037659, … 0.028015]"
209159,"""Window of the Soul (2001)""","""Documentary""","[-0.011909, -0.02803, … -0.005974]"
209163,"""Bad Poems (2018)""","""Comedy|Drama""","[0.002703, 0.005482, … 0.007511]"
